<a href="https://colab.research.google.com/github/Fisev/PZP-Project/blob/Petlu/Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import re
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np



#get files
!wget https://raw.githubusercontent.com/Fisev/PZP-Project/refs/heads/main/data.txt -O data.txt
!wget https://raw.githubusercontent.com/Fisev/PZP-Project/refs/heads/main/stop_words.txt -O stop_words.txt
!pip install pycuda
pattern = re.compile(r'[^a-zA-Z]')
min_length = 4
max_length = 9

from pycuda.compiler import SourceModule
import pycuda.autoinit
import pycuda.driver as cuda

--2024-11-26 18:18:12--  https://raw.githubusercontent.com/Fisev/PZP-Project/refs/heads/main/data.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1257260 (1.2M) [text/plain]
Saving to: ‘data.txt’

data.txt            100%[===================>]   1.20M  --.-KB/s    in 0.04s   

2024-11-26 18:18:12 (28.3 MB/s) - ‘data.txt’ saved [1257260/1257260]

--2024-11-26 18:18:12--  https://raw.githubusercontent.com/Fisev/PZP-Project/refs/heads/main/stop_words.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 100 [text/plain]
Saving to: 

CPU SINGLE TREADED

In [13]:
try:
    # Read lullaby text
    with open("data.txt", "r") as file:
        f_lullaby = file.read().lower()  # Read the entire text as a string

    # Read stop words and create a set for quick lookups
    with open("stop_words.txt", "r") as file:
        f_stopW = set(file.read().lower().split())  # Use a set for faster membership checking

    # Clean the text: remove punctuation
    cleaned_text = pattern.sub(' ', f_lullaby)

    # Split the cleaned text into words
    words = cleaned_text.split()

    # Create a Counter to track word occurrences directly while filtering
    word_counts = Counter()
    filtered_words = []  # List to store filtered words

    # Filter and count words in a single loop
    for word in words:
        cleaned_word = pattern.sub('', word)  # Clean each word
        if min_length <= len(cleaned_word) <= max_length and cleaned_word not in f_stopW:
            word_counts[cleaned_word] += 1
            filtered_words.append(cleaned_word)  # Store the filtered word

    # Get the most and least frequent words
    most_frequent_word, most_frequent_count = word_counts.most_common(1)[0] if word_counts else (None, 0)
    least_frequent_word, least_frequent_count = min(word_counts.items(), key=lambda x: x[1], default=(None, 0))

    # Total number of words after filtering
    total_filtered_words = sum(word_counts.values())

    # Print the results
    print(f"Stop words: {list(f_stopW)}")  # Optionally print stop words for reference
    print(f"Filtered words: {filtered_words}")  # Print the filtered words
    print(f"Most frequent word: '{most_frequent_word}' with {most_frequent_count} occurrences")
    print(f"Least frequent word: '{least_frequent_word}' with {least_frequent_count} occurrences")
    print(f"Total number of filtered words: {total_filtered_words}")

except Exception as e:
    print(f"Reading failed: {e}")

Stop words: ['summer-house', 'barbarians', 'odorous', 'version', 'gutenberg', 'warranty', 'thee', 'ferrule', 'electronic', 'queequeg']
Filtered words: ['project', 'ebook', 'moby', 'dick', 'whale', 'herman', 'melville', 'this', 'ebook', 'anyone', 'anywhere', 'cost', 'with', 'almost', 'copy', 'give', 'away', 'under', 'terms', 'project', 'license', 'included', 'with', 'this', 'ebook', 'online', 'title', 'moby', 'dick', 'whale', 'author', 'herman', 'melville', 'last', 'updated', 'january', 'posting', 'date', 'december', 'ebook', 'release', 'date', 'june', 'language', 'english', 'start', 'this', 'project', 'ebook', 'moby', 'dick', 'whale', 'produced', 'daniel', 'lazarus', 'jonesey', 'moby', 'dick', 'whale', 'herman', 'melville', 'original', 'notes', 'this', 'text', 'etexts', 'from', 'defunct', 'eris', 'project', 'virginia', 'tech', 'from', 'project', 'archives', 'this', 'indebted', 'adelaide', 'library', 'virginia', 'tech', 'resulting', 'etext', 'compared', 'with', 'public', 'domain', 'hard

In [14]:
import pycuda.driver as cuda
import pycuda.autoinit
from pycuda.compiler import SourceModule
import numpy as np
import re
from collections import Counter
import tensorflow as tf

# Example data


# Step 1: Preprocessing on CPU
def preprocess_text(text, stop_words, min_word_length=2):
    # Inicializace

    words = re.findall(r'\b\w+\b', text.lower())  # Tokenizace textu
    word_to_id = {}
    id_to_word = {}
    text_ids = []
    stop_word_ids = set()
    current_id = 0
    data = np.array(list(pattern.sub('', text).encode('utf-8')), dtype=np.int32)

# Vytvoření tensoru pro GPU
    tensor = tf.convert_to_tensor(data)

# Práce s tensorovým polem
    print("Tensor na GPU:", tensor)
    byte_array_text = np.array(list(text.encode('utf-8')) + [0], dtype=np.uint8)
    for word in words + list(stop_words):
        if word not in word_to_id:  # Přidání do mapy, pokud ještě není přidáno
            word_to_id[word] = current_id
            id_to_word[current_id] = word
            current_id += 1

        # Filtrování krátkých slov pouze u textových slov
        if min_length <= len(word) <= max_length:
            text_ids.append(word_to_id[word])

        # Přiřazení ID k stop slovům
        if word in stop_words:
            stop_word_ids.add(word_to_id[word])
    print(byte_array_text)
    # Konverze do NumPy polí
    text_ids = np.array(text_ids, dtype=np.int32)
    stop_word_ids = np.array(list(stop_word_ids), dtype=np.int32)

    return text_ids, stop_word_ids, word_to_id, id_to_word

text_ids, stop_word_ids, word_to_id, id_to_word = preprocess_text(f_lullaby, f_stopW)

# Step 2: CUDA Kernel for Stop Word Filtering and Frequency Counting
kernel_code = """
__global__ void filter_and_count(int *text_ids, int *stop_word_ids, int *word_freq, int text_len, int stop_len) {
    int idx = threadIdx.x + blockIdx.x * blockDim.x;
    if (idx < text_len) {
        int word_id = text_ids[idx];
        bool is_stop_word = false;

        // Check if the word is a stop word
        for (int i = 0; i < stop_len; i++) {
            if (word_id == stop_word_ids[i]) {
                is_stop_word = true;
                break;
            }
        }

        // If not a stop word, increment the frequency count
        if (!is_stop_word) {
            atomicAdd(&word_freq[word_id], 1);
        }
    }
}
"""

# Compile the CUDA kernel
mod = SourceModule(kernel_code)
filter_and_count = mod.get_function("filter_and_count")

# Allocate GPU memory
text_ids_gpu = cuda.mem_alloc(text_ids.nbytes)
stop_word_ids_gpu = cuda.mem_alloc(stop_word_ids.nbytes)
word_freq_gpu = cuda.mem_alloc(len(word_to_id) * np.int32(0).nbytes)

# Initialize word frequency array to zeros
word_freq = np.zeros(len(word_to_id), dtype=np.int32)
cuda.memcpy_htod(text_ids_gpu, text_ids)
cuda.memcpy_htod(stop_word_ids_gpu, stop_word_ids)
cuda.memcpy_htod(word_freq_gpu, word_freq)

# Launch the kernel
block_size = 256
grid_size = (len(text_ids) + block_size - 1) // block_size
filter_and_count(
    text_ids_gpu, stop_word_ids_gpu, word_freq_gpu,
    np.int32(len(text_ids)), np.int32(len(stop_word_ids)),
    block=(block_size, 1, 1), grid=(grid_size, 1)
)

# Copy the results back to the CPU
cuda.memcpy_dtoh(word_freq, word_freq_gpu)

# Step 3: Postprocessing on CPU
# Map word IDs back to words and count frequencies
filtered_word_counts = {id_to_word[idx]: count for idx, count in enumerate(word_freq) if count > 0}

# Identify most and least frequent words
most_frequent_word = max(filtered_word_counts, key=filtered_word_counts.get)
most_frequent_count = filtered_word_counts[most_frequent_word]
least_frequent_word = min(filtered_word_counts, key=filtered_word_counts.get)
least_frequent_count = filtered_word_counts[least_frequent_word]
print(list(text_ids))
# Output results
print(f"Filtered words and frequencies: {filtered_word_counts}")
print(f"Most frequent word: '{most_frequent_word}' with {most_frequent_count} occurrences")
print(f"Least frequent word: '{least_frequent_word}' with {least_frequent_count} occurrences")
print(f"Total number of unique filtered words: {len(filtered_word_counts)}")


Tensor na GPU: tf.Tensor([116 104 101 ... 111 107 115], shape=(967672,), dtype=int32)
[239 187 191 ...  46  10   0]
[1, 2, 3, 5, 6, 8, 10, 11, 12, 3, 16, 17, 20, 22, 23, 28, 30, 31, 33, 34, 1, 2, 35, 36, 22, 12, 3, 37, 2, 40, 5, 6, 8, 41, 10, 11, 42, 43, 44, 46, 47, 48, 49, 51, 3, 52, 53, 48, 54, 55, 56, 57, 58, 12, 1, 2, 3, 5, 6, 8, 59, 60, 61, 62, 5, 6, 8, 10, 11, 63, 66, 12, 67, 70, 72, 74, 75, 1, 76, 77, 72, 1, 2, 78, 12, 80, 82, 85, 86, 76, 77, 80, 88, 89, 91, 22, 92, 93, 94, 28, 80, 67, 96, 102, 104, 105, 106, 107, 108, 109, 110, 111, 113, 114, 115, 116, 113, 118, 119, 120, 121, 126, 127, 130, 131, 22, 132, 134, 22, 138, 139, 140, 141, 142, 143, 131, 144, 145, 146, 147, 148, 149, 150, 115, 151, 152, 153, 154, 155, 8, 156, 158, 160, 161, 163, 164, 165, 167, 23, 168, 169, 171, 172, 173, 167, 175, 176, 8, 179, 12, 180, 181, 72, 182, 183, 184, 185, 186, 187, 8, 189, 72, 193, 194, 196, 197, 199, 200, 201, 202, 203, 204, 205, 184, 206, 208, 209, 210, 8, 211, 8, 57, 212, 213, 214, 215, 

/usr/local/lib/python3.10/dist-packages/google/colab/_variable_inspector.py:27: UserWarning: module in out-of-thread context could not be cleaned up
  globals().clear()
/usr/local/lib/python3.10/dist-packages/google/colab/_variable_inspector.py:27: UserWarning: device_allocation in out-of-thread context could not be cleaned up
  globals().clear()


Spark

In [7]:
from operator import add
from pyspark.sql import SparkSession
import re

pattern = re.compile(r'[^a-zA-Z]')

def normalize_word(word):
    word = pattern.sub('', word)
    return word.lower()
print(list(normalize_word))
spark = SparkSession.builder.appName("WordCount").getOrCreate()

lines = spark.read.text("data.txt").rdd.map(lambda r: r[0])

counts = (
    lines.flatMap(lambda x: x.split(' '))
         .map(normalize_word)
         .filter(lambda word: 4 <= len(word) <= 8)
         .map(lambda x: (x, 1))
         .reduceByKey(add)
)

most_common = counts.sortBy(lambda x: x[1], ascending=False).take(10)

print("slova:")
for word, count in most_common:
    print(f"{word}: {count}")

spark.stop()




<function normalize_word at 0x7d99280feef0>
slova:
that: 2955
with: 1768
this: 1411
from: 1104
whale: 960
have: 770
there: 768
were: 681
they: 660
which: 647
